# Demo — intent parsing end to end

Loads the trained intent-model bundle (training a fresh one first if none
exists), sends utterances through `parse_utterance`, and dumps the
`ParsedUtterance` envelope back as JSON.

Run all cells top-to-bottom. The training procedure lives in
`docs/training.md`.

In [1]:
from pathlib import Path

from gosmart_clinical_nlu import (
    load_intent_models,
    load_training_dir,
    parse_utterance,
    save_bundle,
    train_intent_models,
)

# Work from the repo root whether the kernel starts there or in notebooks/.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent

BUNDLE = ROOT / "models" / "intent_models.zip"
TRAINING_DIR = ROOT / "training"

if not BUNDLE.is_file():
    corpus = load_training_dir(TRAINING_DIR)
    trained = train_intent_models(corpus)
    save_bundle(trained, BUNDLE)
    print(f"no bundle found - trained a fresh one: {BUNDLE}")
else:
    print(f"using existing bundle: {BUNDLE}")

using existing bundle: /Users/manabutokunaga/development/dicom-fhir/gosmart-clinical-nlu/models/intent_models.zip


In [2]:
# Load the whole package: every intent pipeline comes back as one mapping.
models = load_intent_models(BUNDLE)

print("intents:", ", ".join(models.names))
print("spaCy:", models.spacy_version)
print("content hash:", models.content_hash[:16], "...")
for name in models.names:
    spec = models[name].spec
    entity_labels = ", ".join(sorted(spec.entity_map))
    print(
        f"  {name}: label={spec.label} min_confidence={spec.min_confidence} entities=[{entity_labels}]"
    )

assert set(models.names) == {"find_study", "get_report"}
print("OK: bundle loaded")

intents: find_study, get_report
spaCy: 3.8.18
content hash: 3f74a3aa8b4fa9ea ...
  find_study: label=FIND_STUDY min_confidence=0.6 entities=[MODALITY, PATIENT_ID, REL_DATE]
  get_report: label=GET_REPORT min_confidence=0.6 entities=[MODALITY, PATIENT_ID, REL_DATE]
OK: bundle loaded


## Send an utterance

`parse_utterance` runs the text through **every** intent pipeline, dispatches
to the highest-scoring one, extracts its slots, validates them into that
intent's Pydantic model, and wraps everything in a `ParsedUtterance`
envelope. `ref_date` anchors relative dates ("yesterday", "last week").

In [7]:
from datetime import datetime

REF_DATE = datetime(2026, 11, 19)

result = parse_utterance(
    "Show me CT scans for mrn 98765432 last week",
    models,
    ref_date=REF_DATE,
)
print(result.model_dump_json(indent=2))

{
  "status": "matched",
  "intent": "find_study",
  "confidence": 0.9998,
  "threshold": 0.6,
  "candidates": {
    "find_study": 0.9998,
    "get_report": 0.0014
  },
  "missing_required": [],
  "data": {
    "modality": "CT",
    "patient_id": "98765432",
    "study_date": "20261112"
  }
}


In [4]:
# A few more utterances - note the dispatch, the normalizers
# (MRI -> MR DICOM code, relative date -> DICOM DA) and the typed result.
for text in [
    "Find a CT patient 12345678 from yesterday",
    "Get the radiology report for patient 12345678",
    "Read impression on CT for 98765432 from today",
]:
    result = parse_utterance(text, models, ref_date=REF_DATE)
    print(f"{text!r}")
    print(f"  -> {result.intent} ({result.status}, confidence={result.confidence})")
    print(f"     {type(result.data).__name__}: {result.data.model_dump()}")

'Find a CT patient 12345678 from yesterday'
  -> find_study (matched, confidence=0.9999)
     StudyQueryIntent: {'modality': 'CT', 'patient_id': '12345678', 'study_date': '20261118'}
'Get the radiology report for patient 12345678'
  -> get_report (matched, confidence=0.9999)
     ReportQueryIntent: {'modality': None, 'patient_id': '12345678', 'study_date': None}
'Read impression on CT for 98765432 from today'
  -> get_report (matched, confidence=0.9997)
     ReportQueryIntent: {'modality': 'CT', 'patient_id': '98765432', 'study_date': '20261119'}


## The failover signal

`status` is what the caller pattern-matches on:

- `"matched"` - the top intent cleared `min_confidence` and all
  `required_fields` were extracted.
- `"low_confidence"` - hand off to the LLM playbook; `candidates` carries
  every intent's score for context, and best-effort `data` is still returned.

The signal is **data-driven**: with the tiny seed corpus even off-domain
text scores high, so the next cell demonstrates the mechanics by re-training
with an unreachable threshold. In production you grow the examples or raise
`min_confidence` instead (see `docs/training.md` section 6).

In [8]:
corpus = load_training_dir(TRAINING_DIR)
corpus = corpus.model_copy(deep=True)
for item in corpus.intents:
    item.min_confidence = 1.0  # unreachable -> everything comes back low_confidence
strict = train_intent_models(corpus, epochs=50)

result = parse_utterance(
    "Show me MRI scans for mrn 98765432 last week", strict, ref_date=REF_DATE
)
print(result.model_dump_json(indent=2))

if result.status == "low_confidence":
    print(
        "-> caller would fail over to the LLM playbook; candidates:", result.candidates
    )

{
  "status": "low_confidence",
  "intent": "find_study",
  "confidence": 0.9999,
  "threshold": 1.0,
  "candidates": {
    "find_study": 0.9999,
    "get_report": 0.0001
  },
  "missing_required": [],
  "data": {
    "modality": "MR",
    "patient_id": "98765432",
    "study_date": "20261112"
  }
}
-> caller would fail over to the LLM playbook; candidates: {'find_study': 0.9999, 'get_report': 0.0001}


## Result

If every cell above printed its output without a traceback, the full
load -> parse -> dump loop works against the single-file bundle.